# Notebook 03 — Feature Engineering

**Project:** FUSE — Feature Unification for Semantic Exploration

**Objective:** Extract rich NLP features from movie text to build the multi-faceted feature space that powers the FUSE recommendation engine.

**Features to extract:**

| Feature | Method | Input | Dimensionality |
|---------|--------|-------|----------------|
| TF-IDF | sklearn TfidfVectorizer | combined_text | 5000 |
| LDA Topics | sklearn LDA | combined_text | 15 |
| Sentence Embeddings | Sentence-BERT | raw overview | 384 |
| Named Entities | spaCy NER | raw overview | variable |
| Sentiment | TextBlob | raw overview | 2 (polarity, subjectivity) |
| Genre | One-hot encoding | genres column | 19 |

---

## 3.1 — Setup

In [1]:
# !python -m spacy download en_core_web_sm

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.preprocessing import normalize
import scipy.sparse as sp

# ── Optional NLP Libraries ──
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
    print('Sentence-BERT: available')
except ImportError:
    SBERT_AVAILABLE = False
    print('Sentence-BERT: NOT available — will use TruncatedSVD dense embeddings as fallback')

try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    SPACY_AVAILABLE = True
    print('spaCy NER: available')
except (ImportError, OSError):
    SPACY_AVAILABLE = False
    print('spaCy: NOT available — will use keyword-based entity extraction')

try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
    print('TextBlob: available')
except ImportError:
    TEXTBLOB_AVAILABLE = False
    print('TextBlob: NOT available — will use lexicon-based sentiment fallback')

# ── Color Theme ──
GOLD = '#eab308'
BLACK = '#050505'
WHITE = '#ffffff'
GRAY = '#888888'

plt.rcParams.update({
    'figure.facecolor': WHITE, 'axes.facecolor': WHITE,
    'axes.edgecolor': BLACK, 'axes.labelcolor': BLACK,
    'text.color': BLACK, 'xtick.color': BLACK, 'ytick.color': BLACK,
    'font.size': 11, 'axes.titlesize': 14, 'figure.dpi': 100
})

np.random.seed(42)

Sentence-BERT: available
spaCy: NOT available — will use keyword-based entity extraction
TextBlob: available


In [3]:
df = pd.read_csv('../Artifacts/preprocessing/movies_preprocessed.csv')
print(f'Loaded: {df.shape[0]} movies')
print(f'Columns: {list(df.columns)}')

Loaded: 18220 movies
Columns: ['id', 'title', 'vote_average', 'vote_count', 'runtime', 'popularity', 'original_language', 'release_year', 'overview', 'tagline', 'keywords', 'genres', 'overview_clean', 'tagline_clean', 'keywords_clean', 'overview_processed', 'tagline_processed', 'keywords_processed', 'combined_text', 'genre_list']


In [4]:
# Create output directory for features
os.makedirs('../Artifacts/features', exist_ok=True)

## 3.2 — TF-IDF Vectorization

TF-IDF captures term importance across the corpus. We use the `combined_text` field (overview + tagline + keywords, preprocessed) with 5000 features, filtering out extremely common (>95% docs) and rare (<2 docs) terms.

In [5]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    max_df=0.95,
    min_df=2,
    ngram_range=(1, 2),    # unigrams + bigrams
    sublinear_tf=True       # apply log normalization to tf
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_text'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Non-zero entries: {tfidf_matrix.nnz:,} ({tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.2f}% dense)')

TF-IDF matrix shape: (18220, 5000)
Non-zero entries: 595,666 (0.65% dense)


In [ ]:
# Top TF-IDF terms across corpus
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-30:][::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([feature_names[i] for i in top_idx], [mean_tfidf[i] for i in top_idx],
       color=GOLD, edgecolor=BLACK, linewidth=0.3)
ax.set_title('Top 30 TF-IDF Terms (by mean weight across corpus)')
ax.set_ylabel('Mean TF-IDF')
plt.xticks(rotation=50, ha='right')
plt.tight_layout()
plt.show()

In [7]:
# Save TF-IDF
sp.save_npz('../Artifacts/features/tfidf_matrix.npz', tfidf_matrix)
with open('../Artifacts/features/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print('TF-IDF matrix and vectorizer saved.')

TF-IDF matrix and vectorizer saved.


## 3.3 — LDA Topic Modeling

Latent Dirichlet Allocation discovers latent topics in the corpus. Each movie gets a topic distribution vector — a probability distribution over 15 topics. This captures thematic similarity even when movies don't share exact words.

In [8]:
# LDA requires raw counts, not TF-IDF
count_vectorizer = CountVectorizer(
    max_features=5000,
    max_df=0.95,
    min_df=2
)
count_matrix = count_vectorizer.fit_transform(df['combined_text'])

n_topics = 15
lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online',
    batch_size=256
)

lda_matrix = lda_model.fit_transform(count_matrix)
print(f'LDA topic matrix shape: {lda_matrix.shape}')
print(f'Perplexity: {lda_model.perplexity(count_matrix):.1f}')

LDA topic matrix shape: (18220, 15)
Perplexity: 2276.5


In [9]:
# Display top words per topic
cv_feature_names = count_vectorizer.get_feature_names_out()

print('\n=== LDA Topics ===')
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [cv_feature_names[i] for i in topic.argsort()[-10:][::-1]]
    print(f'\nTopic {topic_idx:2d}: {" | ".join(top_words)}')


=== LDA Topics ===

Topic  0: future | space | time | prison | world | earth | scientist | ship | planet | year

Topic  1: town | small | get | life | friend | one | find | dog | time | back

Topic  2: new | city | york | england | london | journalist | reporter | paris | france | life

Topic  3: car | island | sport | road | trip | race | team | crash | park | desert

Topic  4: alien | monster | creature | human | animal | fight | world | battle | giant | must

Topic  5: house | evil | horror | game | death | ghost | dead | video | night | supernatural

Topic  6: love | woman | life | relationship | friend | director | man | one | couple | marriage

Topic  7: school | high | based | girl | student | power | teen | friend | vampire | teenage

Topic  8: killer | murder | revenge | art | serial | agent | secret | kill | martial | one

Topic  9: police | christmas | cop | los | angeles | holiday | detective | california | criminal | gang

Topic 10: family | father | child | relationship 

In [ ]:
# Visualize topic distribution across the corpus
topic_means = lda_matrix.mean(axis=0)
topic_labels = [f'Topic {i}' for i in range(n_topics)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(topic_labels, topic_means, color=GOLD, edgecolor=BLACK, linewidth=0.5)
ax.set_title('Average Topic Proportion Across All Movies')
ax.set_ylabel('Mean Proportion')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Topic distribution heatmap for first 20 movies
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(lda_matrix[:20], cmap='YlOrBr', aspect='auto')
ax.set_yticks(range(20))
ax.set_yticklabels(df['title'].iloc[:20], fontsize=8)
ax.set_xticks(range(n_topics))
ax.set_xticklabels([f'T{i}' for i in range(n_topics)])
ax.set_title('Topic Distribution — First 20 Movies')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [12]:
# Save LDA
np.save('../Artifacts/features/lda_matrix.npy', lda_matrix)
with open('../Artifacts/features/lda_model.pkl', 'wb') as f:
    pickle.dump(lda_model, f)
with open('../Artifacts/features/count_vectorizer.pkl', 'wb') as f:
    pickle.dump(count_vectorizer, f)
print('LDA model and topic matrix saved.')

LDA model and topic matrix saved.


## 3.4 — Sentence Embeddings

Sentence-BERT produces dense semantic vectors that capture the meaning of entire sentences. We embed the raw overview (natural English) rather than the preprocessed tokens.

**If Sentence-BERT is unavailable**, we use TruncatedSVD on the TF-IDF matrix to produce dense 100-dimensional embeddings as a fallback. This is a common LSA (Latent Semantic Analysis) approach.

In [13]:
if SBERT_AVAILABLE:
    print('Encoding overviews with Sentence-BERT (all-MiniLM-L6-v2)...')
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    sbert_matrix = sbert_model.encode(
        df['overview'].tolist(),
        show_progress_bar=True,
        batch_size=128
    )
    sbert_matrix = np.array(sbert_matrix)
    print(f'SBERT embeddings shape: {sbert_matrix.shape}')
else:
    print('Using TruncatedSVD (LSA) as dense embedding fallback...')
    n_components = 100
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    sbert_matrix = svd.fit_transform(tfidf_matrix)
    sbert_matrix = normalize(sbert_matrix)  # L2 normalize for cosine similarity
    explained = svd.explained_variance_ratio_.sum()
    print(f'SVD embeddings shape: {sbert_matrix.shape}')
    print(f'Explained variance: {explained*100:.1f}%')

Encoding overviews with Sentence-BERT (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9001.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 143/143 [01:49<00:00,  1.30it/s]

SBERT embeddings shape: (18220, 384)


In [ ]:
# Visualize embedding space with PCA (2D projection)
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
embedding_2d = pca.fit_transform(sbert_matrix)

# Color by dominant genre
genre_list = df['genres'].str.split(', ').apply(lambda x: x[0] if x else 'Unknown')
top5_genres = genre_list.value_counts().head(5).index.tolist()
mask = genre_list.isin(top5_genres)
colors_map = {g: c for g, c in zip(top5_genres, [GOLD, BLACK, '#c49b08', GRAY, '#444444'])}

fig, ax = plt.subplots(figsize=(10, 8))
for genre in top5_genres:
    idx = genre_list == genre
    ax.scatter(embedding_2d[idx, 0], embedding_2d[idx, 1],
              s=3, alpha=0.3, label=genre, color=colors_map[genre])
ax.set_title('Embedding Space (PCA 2D) — Top 5 Primary Genres')
ax.legend(markerscale=5, fontsize=9)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.show()

In [15]:
np.save('../Artifacts/features/embedding_matrix.npy', sbert_matrix)
print(f'Sentence embeddings saved: {sbert_matrix.shape}')

Sentence embeddings saved: (18220, 384)


## 3.5 — Named Entity Recognition (NER)

We extract named entities (persons, locations, organizations) from overviews to capture specific references that might connect movies.

**If spaCy is unavailable**, we extract proper-noun-like patterns from the keywords field as a proxy.

In [16]:
import re as regex_module

if SPACY_AVAILABLE:
    print('Extracting named entities with spaCy...')
    entity_types = ['PERSON', 'GPE', 'ORG', 'LOC', 'EVENT']
    
    def extract_entities(text):
        if not isinstance(text, str) or not text.strip():
            return {}
        doc = nlp(text)
        entities = {}
        for ent in doc.ents:
            if ent.label_ in entity_types:
                key = f'{ent.label_}:{ent.text.lower()}'
                entities[key] = entities.get(key, 0) + 1
        return entities
    
    # Process in batches for memory
    ner_results = []
    batch_size = 1000
    for i in range(0, len(df), batch_size):
        batch = df['overview'].iloc[i:i+batch_size].tolist()
        for text in batch:
            ner_results.append(extract_entities(str(text)))
        if (i + batch_size) % 5000 == 0:
            print(f'  Processed {min(i+batch_size, len(df))}/{len(df)}...')
    
    df['entities'] = ner_results
else:
    print('Using keyword-based entity extraction...')
    # Extract from keywords: location/person-like terms
    def extract_keyword_entities(keywords_str):
        if not isinstance(keywords_str, str) or not keywords_str.strip():
            return {}
        keywords = [k.strip().lower() for k in keywords_str.split(',')]
        entities = {f'KW:{kw}': 1 for kw in keywords if kw}
        return entities
    
    df['entities'] = df['keywords'].apply(extract_keyword_entities)

print(f'Entities extracted for {len(df)} movies.')

# Count most common entities
all_entities = Counter()
for ent_dict in df['entities']:
    if isinstance(ent_dict, dict):
        all_entities.update(ent_dict)
    
print(f'Total unique entities: {len(all_entities)}')
print(f'Top 15 entities:')
for ent, count in all_entities.most_common(15):
    print(f'  {ent}: {count}')

Using keyword-based entity extraction...
Entities extracted for 18220 movies.
Total unique entities: 19147
Top 15 entities:
  KW:based on novel or book: 1457
  KW:woman director: 1097
  KW:murder: 1067
  KW:new york city: 748
  KW:california: 636
  KW:sequel: 622
  KW:based on true story: 558
  KW:revenge: 547
  KW:biography: 516
  KW:duringcreditsstinger: 509
  KW:england: 500
  KW:christmas: 491
  KW:short film: 478
  KW:musical: 392
  KW:los angeles: 374


In [ ]:
# Visualize top entities
top_ents = all_entities.most_common(25)
ent_names = [e[0] for e in top_ents]
ent_counts = [e[1] for e in top_ents]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(ent_names[::-1], ent_counts[::-1], color=GOLD, edgecolor=BLACK, linewidth=0.3)
ax.set_title('Top 25 Named Entities Across Movie Overviews')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

In [18]:
# Save entities
with open('../Artifacts/features/entities.pkl', 'wb') as f:
    pickle.dump(df['entities'].tolist(), f)
print('Entities saved.')

Entities saved.


## 3.6 — Sentiment Analysis

We extract polarity (positive/negative tone) and subjectivity (objective/subjective) from movie overviews. This captures the emotional tone of the writing.

**If TextBlob is unavailable**, we use a simple positive/negative word count ratio as a proxy.

In [19]:
if TEXTBLOB_AVAILABLE:
    print('Computing sentiment with TextBlob...')
    sentiments = df['overview'].apply(lambda x: TextBlob(str(x)).sentiment)
    df['polarity'] = sentiments.apply(lambda s: s.polarity)
    df['subjectivity'] = sentiments.apply(lambda s: s.subjectivity)
else:
    print('Using lexicon-based sentiment fallback...')
    # Simple positive/negative word lists
    pos_words = {'good','great','best','love','beautiful','wonderful','amazing','excellent',
                 'brilliant','fantastic','outstanding','perfect','happy','joy','hope','hero',
                 'brave','triumph','win','success','save','rescue','peace','friend'}
    neg_words = {'bad','worst','terrible','horrible','evil','dark','death','kill','murder',
                 'war','hate','fear','dangerous','threat','corrupt','destroy','fail','lost',
                 'crime','violent','suffer','pain','struggle','attack','enemy','revenge'}
    
    def simple_sentiment(text):
        if not isinstance(text, str):
            return 0.0, 0.5
        words = text.lower().split()
        n = max(len(words), 1)
        pos_count = sum(1 for w in words if w in pos_words)
        neg_count = sum(1 for w in words if w in neg_words)
        polarity = (pos_count - neg_count) / n
        subjectivity = (pos_count + neg_count) / n
        return polarity, min(subjectivity, 1.0)
    
    results = df['overview'].apply(simple_sentiment)
    df['polarity'] = results.apply(lambda x: x[0])
    df['subjectivity'] = results.apply(lambda x: x[1])

print(f'Polarity    — mean: {df["polarity"].mean():.3f}, std: {df["polarity"].std():.3f}')
print(f'Subjectivity — mean: {df["subjectivity"].mean():.3f}, std: {df["subjectivity"].std():.3f}')

Computing sentiment with TextBlob...
Polarity    — mean: 0.033, std: 0.231
Subjectivity — mean: 0.463, std: 0.231


In [ ]:
# Sentiment distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['polarity'], bins=50, color=GOLD, edgecolor=BLACK, linewidth=0.3)
axes[0].set_title('Polarity Distribution')
axes[0].set_xlabel('Polarity (negative ← 0 → positive)')
axes[0].axvline(0, color=BLACK, linestyle='--', linewidth=0.8)

axes[1].hist(df['subjectivity'], bins=50, color=GOLD, edgecolor=BLACK, linewidth=0.3)
axes[1].set_title('Subjectivity Distribution')
axes[1].set_xlabel('Subjectivity (objective ← 0 → subjective)')

plt.tight_layout()
plt.show()

In [ ]:
# Sentiment by genre
genre_sent = df[['genres', 'polarity']].copy()
genre_sent['genres'] = genre_sent['genres'].str.split(', ')
genre_sent = genre_sent.explode('genres')

genre_polarity = genre_sent.groupby('genres')['polarity'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genre_polarity.index, genre_polarity.values, color=GOLD, edgecolor=BLACK, linewidth=0.5)
ax.set_title('Average Polarity by Genre')
ax.set_xlabel('Mean Polarity')
ax.axvline(0, color=GRAY, linestyle='--')
plt.tight_layout()
plt.show()

In [22]:
sentiment_matrix = df[['polarity', 'subjectivity']].values
np.save('../Artifacts/features/sentiment_matrix.npy', sentiment_matrix)
print(f'Sentiment matrix saved: {sentiment_matrix.shape}')

Sentiment matrix saved: (18220, 2)


## 3.7 — Genre One-Hot Encoding

In [23]:
# Parse genre lists
import ast
try:
    genre_lists = df['genre_list'].apply(ast.literal_eval)
except:
    genre_lists = df['genres'].str.split(', ')

all_genres = sorted(set(g.strip() for gl in genre_lists for g in gl))
print(f'Genres ({len(all_genres)}): {all_genres}')

# One-hot encode
genre_matrix = np.zeros((len(df), len(all_genres)), dtype=np.float32)
for i, gl in enumerate(genre_lists):
    for g in gl:
        g = g.strip()
        if g in all_genres:
            genre_matrix[i, all_genres.index(g)] = 1.0

print(f'Genre matrix shape: {genre_matrix.shape}')
print(f'Avg genres per movie: {genre_matrix.sum(axis=1).mean():.1f}')

Genres (19): ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Genre matrix shape: (18220, 19)
Avg genres per movie: 2.5


In [ ]:
# Genre frequency
genre_freq = genre_matrix.sum(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(all_genres, genre_freq, color=GOLD, edgecolor=BLACK, linewidth=0.5)
ax.set_title('Genre Frequency (One-Hot Encoded)')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [25]:
np.save('../Artifacts/features/genre_matrix.npy', genre_matrix)
with open('../Artifacts/features/genre_names.pkl', 'wb') as f:
    pickle.dump(all_genres, f)
print('Genre matrix saved.')

Genre matrix saved.


## 3.8 — Feature Summary & Alignment Check

We verify that all feature matrices are aligned (same number of rows) and save a metadata file.

In [26]:
print('=== Feature Matrix Summary ===')
print(f'Movies:              {len(df)}')
print(f'TF-IDF:              {tfidf_matrix.shape}')
print(f'LDA Topics:          {lda_matrix.shape}')
print(f'Sentence Embeddings: {sbert_matrix.shape}')
print(f'Sentiment:           {sentiment_matrix.shape}')
print(f'Genre Encoding:      {genre_matrix.shape}')

# Alignment check
assert tfidf_matrix.shape[0] == len(df)
assert lda_matrix.shape[0] == len(df)
assert sbert_matrix.shape[0] == len(df)
assert sentiment_matrix.shape[0] == len(df)
assert genre_matrix.shape[0] == len(df)
print('\n✓ All feature matrices aligned.')

=== Feature Matrix Summary ===
Movies:              18220
TF-IDF:              (18220, 5000)
LDA Topics:          (18220, 15)
Sentence Embeddings: (18220, 384)
Sentiment:           (18220, 2)
Genre Encoding:      (18220, 19)

✓ All feature matrices aligned.


In [ ]:
# Visualize feature dimensions
feature_dims = {
    'TF-IDF': tfidf_matrix.shape[1],
    'LDA Topics': lda_matrix.shape[1],
    'Embeddings': sbert_matrix.shape[1],
    'Sentiment': sentiment_matrix.shape[1],
    'Genre': genre_matrix.shape[1]
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(feature_dims.keys(), feature_dims.values(), color=GOLD, edgecolor=BLACK, linewidth=0.5)
for bar, v in zip(bars, feature_dims.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(v), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Feature Dimensionality per Module')
ax.set_ylabel('Dimensions')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [28]:
# Save movie metadata (id, title, etc.) aligned with features
df[['id', 'title', 'vote_average', 'vote_count', 'popularity', 'release_year',
    'overview', 'tagline', 'keywords', 'genres', 'combined_text',
    'polarity', 'subjectivity']].to_csv('../Artifacts/features/movies_with_features.csv', index=False)
print('Movie metadata saved alongside features.')

Movie metadata saved alongside features.


## Summary

**Features extracted:**

| Feature | Shape | Method |
|---------|-------|--------|
| TF-IDF | (N, 5000) | Unigrams + bigrams, sublinear TF |
| LDA Topics | (N, 15) | Latent Dirichlet Allocation |
| Embeddings | (N, 384/100) | SBERT or TruncatedSVD fallback |
| Sentiment | (N, 2) | Polarity + subjectivity |
| Genre | (N, 19) | One-hot encoding |

All matrices are saved in `../Artifacts/features/` and are row-aligned.

**Next:** Notebook 04 — FUSE Recommendation Engine